# 0605 — cGAN Denormalization: Injecting Semantic Names into the Synthetic Manifold

Denormalizes the ~1M-row synthetic manifold produced by the cGAN: builds a master zone dictionary (manually-drawn P_ polygons + HDBSCAN C_ clusters), validates it against what actually survives in the manifold and in real data, then joins product/pickup/dropoff semantic names onto the manifold in Spark and writes the enriched result to BigQuery as `synthetic_manifold_v8_enriched` — the table `0606`/`0607` read as source.

```mermaid
flowchart TD
    A["local parquet: 260426_cGAN_manifold_v8.parquet\n(~1M synthetic rows)"] --> P1["Phase 1\nSpark setup:\nsession init, load manifold"]
    P1 --> P2["Phase 2\nBigQuery destination config:\nproject/dataset + GCS staging"]
    P2 --> B["Phase 3\nMaster zone dictionary:\n89 zones -> purge -> 67 survivors"]
    B --> C["Phase 4\nDictionary persistence:\nproduct + dropoff + pickup parquets"]
    A --> D["Phase 5\nTriple star schema join:\ninject product/pickup/dropoff names"]
    C --> D
    D --> E["Phase 6\nWrite to BigQuery:\npienza_big.synthetic_manifold_v8_enriched"]
    E --> F["validated by 0606/0607\nas their source table"]

    classDef default stroke:#21918c,stroke-width:2px;
    linkStyle default stroke:#21918c,stroke-width:2px;
```


## Phase 1 — Spark setup

Initializes the Spark session (with the BigQuery connector) and loads the local synthetic manifold parquet.

#### 1.1 — Spark session initialization

In [1]:
import os
import warnings
from itertools import chain

import findspark
import geopandas as gpd
import pandas as pd
from google.cloud import bigquery
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# --- Credentials ---
SA_PATH = "/workspaces/pienza/secrets/service-account.json"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = SA_PATH

# --- Locate Spark ---
findspark.init()

# --- Build the engine ---
# Note: 4g RAM to comfortably handle the million-row manifold
spark = SparkSession.builder \
    .appName("Pienza_Denormalization") \
    .config("spark.jars.packages", "com.google.cloud.spark:spark-bigquery-with-dependencies_2.12:0.34.0") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# --- Hadoop config (GCS-BigQuery bridge) ---
spark._jsc.hadoopConfiguration().set("google.cloud.auth.service.account.enable", "true")
spark._jsc.hadoopConfiguration().set("google.cloud.auth.service.account.json.keyfile", SA_PATH)

print("--- Spark started ---")
print(f"Version: {spark.version}")
print("BigQuery connector loaded and authenticated via service account.")


26/07/14 02:52:35 WARN Utils: Your hostname, codespaces-ff38a8 resolves to a loopback address: 127.0.0.1; using 10.0.5.112 instead (on interface eth0)
26/07/14 02:52:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/vscode/.local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/vscode/.ivy2/cache
The jars for the packages stored in: /home/vscode/.ivy2/jars
com.google.cloud.spark#spark-bigquery-with-dependencies_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d449662d-36ae-4bbe-b985-c11bd5439074;1.0
	confs: [default]
	found com.google.cloud.spark#spark-bigquery-with-dependencies_2.12;0.34.0 in central
:: resolution report :: resolve 184ms :: artifacts dl 4ms
	:: modules in use:
	com.google.cloud.spark#spark-bigquery-with-dependencies_2.12;0.34.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	-----------------------------------------------------------------

--- Spark started ---
Version: 3.5.1
BigQuery connector loaded and authenticated via service account.


#### 1.2 — Load the local manifold (V8)

Loads the cGAN's synthetic manifold parquet into Spark and registers it as a temp view.

In [2]:
# Path to the dataset in the local workspace (GAN V8 output)
PARQUET_SOURCE_PATH = "/workspaces/pienza/data/dumped_files/260426_cGAN_manifold_v8.parquet"

print(f"Loading master dataset from: {PARQUET_SOURCE_PATH} ...")

try:
    if not os.path.exists(PARQUET_SOURCE_PATH):
        raise FileNotFoundError(f"File not found at {PARQUET_SOURCE_PATH}")

    df_manifold = spark.read.parquet(PARQUET_SOURCE_PATH)

    # Register as a temp view for SQL
    df_manifold.createOrReplaceTempView("simulacion_manifold")

    print("--- Manifold V8 loaded ---")
    print(f"Total rows: {df_manifold.count():,}")
    df_manifold.printSchema()

except Exception as e:
    print("ERROR: could not read the file. Check the path.")
    print(e)


Loading master dataset from: /workspaces/pienza/data/dumped_files/260426_cGAN_manifold_v8.parquet ...
--- Manifold V8 loaded ---
Total rows: 1,010,001
root
 |-- upfront_fare: float (nullable = true)
 |-- est_trip_time_sec: float (nullable = true)
 |-- est_trip_dist_km: float (nullable = true)
 |-- time_to_pickup_sec: float (nullable = true)
 |-- dist_to_pickup_km: float (nullable = true)
 |-- hour_of_day: string (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- product_category_fk: string (nullable = true)
 |-- dropoff_zone_id: string (nullable = true)
 |-- pickup_zone_id: string (nullable = true)
 |-- reason_primary_fk: string (nullable = true)



## Phase 2 — BigQuery destination config

Sets up the project/dataset target and the GCS staging bucket Spark needs to bridge writes into BigQuery.

#### 2.1 — BigQuery destination & GCS staging

In [3]:
# --- Project variables ---
GCP_PROJECT_ID = "drivers-dilemma"
BQ_DATASET = "pienza_big"

# --- GCS staging config ---
# Spark needs a temporary bucket to stage data before loading it into BigQuery
STAGING_BUCKET = "pienza-streamlit"

spark.conf.set("temporaryGcsBucket", STAGING_BUCKET)

print("Cloud architecture configured:")
print(f"   Destination: {GCP_PROJECT_ID}.{BQ_DATASET}")
print(f"   GCS staging: {STAGING_BUCKET}")


Cloud architecture configured:
   Destination: drivers-dilemma.pienza_big
   GCS staging: pienza-streamlit


## Phase 3 — Master zone dictionary

Builds the unified P_/C_/Unassigned zone dictionary (89 zones), then validates it in three passes: which zones actually survive in the manifold (67), which got dropped (22), and how those 67 compose against real-world observation counts.

#### 3.1 — Universal lookup table (bulletproof version)

Builds the P_ axis (manually-named polygons — some names are concatenations of merged polygons) and the C_ axis (HDBSCAN cluster names from `silver_palette` in BigQuery), then unions them with an `Unassigned` fallback into the master dictionary (89 zones).

In [4]:
print("Forging the Pienza lookup table (maximum-robustness protocol)...")

# --- 1. Identity config (text project ID, GCP resource standard) ---
BILLING_PROJECT_ID = "drivers-dilemma"
DATASET_REALIDAD = "pienza_mini"

# --- 2. Human dictionary (P_ axis) ---
# Some semantic_name values are concatenations of multiple merged polygons
p_axis_zone_names = [
    (0, "santa_fe_bosques_de__santa_fe_cumbres_de__santa_fe_tec"),
    (1, "santa_fe_centro_comercial"),
    (2, "carretera_al_olivo__carretera_libre__cruce_echanove__vistahermosa"),
    (3, "bosques_pabellon__el_olivo__loma_de_la_palma"),
    (4, "agwa_bezares__reforma_bnp"), (5, "ahuehuetes_norte__de_los_bosques"),
    (6, "interlomas_haciendas__jesus_del_monte"), (7, "blvrd_anahuac__universidad_anahuac"),
    (8, "ave_club_de_golf_lomas__interlomas_magnocentro__vialidad_de_la_barranca"),
    (9, "santa_fe_ibero"), (10, "lomas_altas__nodo_reforma_palmas__reforma_regina"),
    (11, "ahuehuetes_sur"), (12, "tamarindos"), (13, "santa_fe_quintana__sante_fe_patio"),
    (14, "bosque_real__lomas_country_club"), (15, "herradura_conscripto"),
    (16, "de_las_fuentes__tecamachalco"), (17, "lomas_barrilaco__lomas_olimpo__nodo_monte_libano"),
    (18, "lomas_prado_norte__lomas_trastevere"), (19, "lomas_virreyes"),
    (20, "lomas_fc_cuernavaca"), (21, "fuentes_casino__sedena__tecamachalco"),
    (22, "palmas_jp_morgan"), (23, "bondojito_asf__bosque_2__bosque_3"),
    (24, "bosque_1__campo_marte"), (25, "roma_condesa_2"), (26, "rios"),
    (27, "roma_condesa_1"), (28, "juarez_soho_house"), (29, "anzures"),
    (30, "anahuac_1__bahias__frontera_polanco"), (31, "sotelo"),
    (32, "carso_antara_miyana"), (33, "irrigacion__polanco_uber_hq"),
    (34, "juarez_rosa"), (35, "lagos"), (36, "polanco_grupo_mexico__polanco_palacio"),
    (37, "polanco_gandhi"), (38, "polanco_5"), (39, "polanco_parque_lincoln"),
    (40, "polanco_parroquia"), (41, "santa_fe_colegios")
]
df_dict_p = spark.createDataFrame(p_axis_zone_names, ["id_raw", "semantic_name"]) \
                 .withColumn("zone_key", F.concat(F.lit("P_"), F.col("id_raw").cast("string"))) \
                 .select("zone_key", "semantic_name")

# --- 3. Machine dictionary (C_ axis) ---
try:
    print(f"Fetching cluster names from {DATASET_REALIDAD}.silver_palette...")
    # parentProject tells Google who to bill for the scanned bytes
    df_silver_palette = spark.read.format("bigquery") \
        .option("table", f"{BILLING_PROJECT_ID}.{DATASET_REALIDAD}.silver_palette") \
        .option("parentProject", BILLING_PROJECT_ID) \
        .load()

    df_dict_c = df_silver_palette \
        .select("dropoff_hdbscan_id", "dropoff_hdbscan_name") \
        .distinct() \
        .filter(F.col("dropoff_hdbscan_id") != -1) \
        .withColumn("zone_key", F.concat(F.lit("C_"), F.col("dropoff_hdbscan_id").cast("string"))) \
        .select("zone_key", F.col("dropoff_hdbscan_name").alias("semantic_name"))

    print(f"   Machine metadata fetched: {df_dict_c.count()} clusters.")

except Exception as e:
    print("CRITICAL FAILURE reading BigQuery.")
    print("Possible causes: 'drivers-dilemma' is wrong, or parentProject is missing.")
    raise e

# --- 4. Final unification ---
df_unassigned = spark.createDataFrame([("Unassigned", "unassigned_area")], ["zone_key", "semantic_name"])

df_master_dictionary = df_dict_p.union(df_dict_c).union(df_unassigned)

print(f"Master dictionary unified: {df_master_dictionary.count()} zones.")
df_master_dictionary.orderBy("zone_key").show(10, truncate=False)


Forging the Pienza lookup table (maximum-robustness protocol)...
Fetching cluster names from pienza_mini.silver_palette...


   Machine metadata fetched: 46 clusters.


Master dictionary unified: 89 zones.


+--------+-----------------------+
|zone_key|semantic_name          |
+--------+-----------------------+
|C_-2    |missing_coordinates    |
|C_0     |viaducto_tlalpan       |
|C_1     |terminal_1_aicm        |
|C_10    |observatorio           |
|C_11    |sotelo_san_esteban     |
|C_12    |barranca_del_muerto    |
|C_13    |haciendas_san_fernando |
|C_14    |vialidad_de_la_barranca|
|C_15    |interlomas_magnocentro |
|C_16    |santa_fe_itesm         |
+--------+-----------------------+
only showing top 10 rows



#### 3.2 — Full geographic reconnaissance (the 89 zones)

Displays the complete dictionary for manual audit.

In [5]:
print("MASTER ZONE LISTING (89 SEMANTIC IDENTITIES)")
print("-" * 80)

# Convert to Pandas for display
df_full_audit = df_master_dictionary.orderBy("zone_key").toPandas()

# Don't truncate the long concatenated names
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', None)

display(df_full_audit)

print("-" * 80)
print(f"Total: {len(df_full_audit)} records ready for the million-row join.")


MASTER ZONE LISTING (89 SEMANTIC IDENTITIES)
--------------------------------------------------------------------------------


,zone_key,semantic_name
0,C_-2,missing_coordinates
1,C_0,viaducto_tlalpan
2,C_1,terminal_1_aicm
3,C_10,observatorio
4,C_11,sotelo_san_esteban
5,C_12,barranca_del_muerto
6,C_13,haciendas_san_fernando
7,C_14,vialidad_de_la_barranca
8,C_15,interlomas_magnocentro
9,C_16,santa_fe_itesm


--------------------------------------------------------------------------------
Total: 89 records ready for the million-row join.


#### 3.3 — Geographic purge (non-destructive)

Filters the 89-zone dictionary down to only the zones that actually appear in the manifold (67 survive).

In [6]:
print("Identifying active zones in the manifold...")

# 1. Keep a copy of the full dictionary (89 rows) before filtering
df_master_dictionary_full = df_master_dictionary

# 2. Identify the IDs that actually exist in the manifold (1M rows)
df_active_keys = df_manifold.select("dropoff_zone_id").distinct()

# 3. Build the curated dictionary (67 rows)
df_dict_curated = df_master_dictionary_full.join(
    df_active_keys,
    df_master_dictionary_full.zone_key == df_active_keys.dropoff_zone_id,
    "inner"
).select("zone_key", "semantic_name")

print(f"   Filtering complete: 89 -> {df_dict_curated.count()} live zones.")

# Overwrite the main pointer for the final join
df_master_dictionary = df_dict_curated


Identifying active zones in the manifold...


   Filtering complete: 89 -> 67 live zones.


#### 3.4 — Shadow audit (identifying deleted zones)

Reports which of the 89 dictionary zones did not survive into the manifold (22 dropped).

In [7]:
print("Investigating the 22 deleted zones...")

# Find what's in the full dictionary but not in the curated one
df_deleted_zones = df_master_dictionary_full.join(
    df_active_keys,
    df_master_dictionary_full.zone_key == df_active_keys.dropoff_zone_id,
    "left_anti"
)

df_deleted_report = df_deleted_zones.orderBy("zone_key").toPandas()

print("\nExclusion list (shadow / absorbed zones):")
print("-" * 80)
if len(df_deleted_report) > 0:
    display(df_deleted_report)
else:
    print("No deleted zones found.")

print("-" * 80)
print("Audit finished. These zones will no longer contaminate the manifold.")


Investigating the 22 deleted zones...



Exclusion list (shadow / absorbed zones):
--------------------------------------------------------------------------------


,zone_key,semantic_name
0,C_-2,missing_coordinates
1,C_14,vialidad_de_la_barranca
2,C_20,tamarindos
3,C_21,nodo_constituyentes_reforma
4,C_24,santa_fe_core
5,C_25,pabellon_bosques
6,C_26,duraznos
7,C_27,chamizal
8,C_28,lomas_anahuac
9,C_31,san_miguel_chapultepec


--------------------------------------------------------------------------------
Audit finished. These zones will no longer contaminate the manifold.


#### 3.5 — Ghost recovery (forced comparison)

Re-derives the full P_+C_+Unassigned universe from scratch and anti-joins it against the manifold's active keys, as an independent cross-check of the purge in 3.3-3.4.

In [8]:
print("Recovering the 22 ghosts from first principles...")

# 1. Reconstruct the full dictionary (P_ + C_)
df_reconstructed_full = df_dict_p.union(df_dict_c).union(df_unassigned)

# 2. Identify the IDs that actually exist in the manifold (1M)
df_active_manifold_keys = df_manifold.select("dropoff_zone_id").distinct()

# 3. Anti-join: what was in the dictionary but never reached the manifold?
df_ghosts = df_reconstructed_full.join(
    df_active_manifold_keys,
    df_reconstructed_full.zone_key == df_active_manifold_keys.dropoff_zone_id,
    "left_anti"
)

ghosts_pd = df_ghosts.orderBy("zone_key").toPandas()

print(f"\nPurged zones ({len(ghosts_pd)} records):")
print("-" * 80)
if len(ghosts_pd) > 0:
    display(ghosts_pd)
    print("\nAnalysis: these zones were either absorbed by the P_ polygons")
    print("   or the GAN determined their probability of occurrence is zero.")
else:
    print("Something is still off in the reference. Proceeding to the join to keep momentum.")

# Make sure the curated version is used for the final join
df_master_dictionary = df_reconstructed_full.join(
    df_active_manifold_keys,
    df_reconstructed_full.zone_key == df_active_manifold_keys.dropoff_zone_id,
    "inner"
).select("zone_key", "semantic_name")


Recovering the 22 ghosts from first principles...



Purged zones (22 records):
--------------------------------------------------------------------------------


,zone_key,semantic_name
0,C_-2,missing_coordinates
1,C_14,vialidad_de_la_barranca
2,C_20,tamarindos
3,C_21,nodo_constituyentes_reforma
4,C_24,santa_fe_core
5,C_25,pabellon_bosques
6,C_26,duraznos
7,C_27,chamizal
8,C_28,lomas_anahuac
9,C_31,san_miguel_chapultepec



Analysis: these zones were either absorbed by the P_ polygons
   or the GAN determined their probability of occurrence is zero.


#### 3.6 — Real-world composition audit (the 67 survivors)

Joins the curated dictionary against real observation counts (from `v_ML_Supervised`) to see how each surviving zone's frequency compares between the synthetic and real worlds.

In [9]:
print("Starting forensic audit of reality...")

# --- 1. Re-inject the canonical map ---
id_map = {
    -1: -1, 41: 0, 42: 0, 46: 0, 43: 1, 65: 2, 62: 2, 44: 2, 36: 2, 49: 3, 52: 3,
    35: 3, 50: 4, 58: 4, 25: 5, 31: 5, 63: 6, 39: 6, 51: 7, 33: 7, 37: 8, 53: 8,
    48: 8, 60: 9, 57: 10, 12: 10, 32: 10, 24: 11, 40: 12, 45: 13, 59: 13, 61: 14,
    38: 14, 34: 15, 30: 16, 66: 16, 17: 17, 14: 17, 22: 17, 16: 18, 13: 18, 11: 19,
    15: 20, 21: 21, 20: 21, 19: 21, 18: 22, 47: 23, 55: 23, 56: 23, 54: 24, 64: 24,
    71: 25, 9: 26, 70: 27, 69: 28, 8: 29, 6: 30, 7: 30, 23: 30, 3: 31, 2: 32,
    4: 33, 29: 33, 68: 34, 5: 35, 27: 36, 28: 36, 1: 37, 10: 38, 0: 39, 26: 40, 67: 41
}

try:
    # --- 2. Load from BigQuery ---
    # Read via the bigquery client (pandas) instead of Spark's native BigQuery
    # connector: v_ML_Supervised is a view, and the connector's view-materialization
    # path (viewsEnabled + materializationDataset) has shown flaky NOT_FOUND errors
    # on the temp _bqc_ table it creates. Going through pandas sidesteps that entirely.
    bq_client_local = bigquery.Client(project=BILLING_PROJECT_ID)
    query_real = f"""
        SELECT offer_id, dropoff_polygon_id, dropoff_hdbscan_id
        FROM `{BILLING_PROJECT_ID}.{DATASET_REALIDAD}.v_ML_Supervised`
    """
    pdf_real_raw = bq_client_local.query(query_real).to_dataframe()
    df_real_raw = spark.createDataFrame(pdf_real_raw)

    # --- 3. Spatial processing ---
    # Convert id_map into a Spark Map column
    mapping_expr = F.create_map([F.lit(x) for x in chain(*id_map.items())])

    # Apply coalesce + mapping (Spark 3.0+ syntax)
    df_real_mapped = df_real_raw.withColumn(
        "id_agrupado",
        F.coalesce(mapping_expr[F.col("dropoff_polygon_id").cast("int")], F.lit(-1))
    )

    # Build the zone_key
    df_real_final = df_real_mapped.withColumn("zone_key",
        F.when(F.col("id_agrupado") >= 0, F.concat(F.lit("P_"), F.col("id_agrupado").cast("string")))
         .when(F.col("dropoff_hdbscan_id") > -1, F.concat(F.lit("C_"), F.col("dropoff_hdbscan_id").cast("string")))
         .otherwise("Unassigned")
    )

    # Real frequency count
    df_real_counts = df_real_final.groupBy("zone_key").count().withColumnRenamed("count", "n_obs_real")

    # --- 4. Join and null cleanup ---
    df_final_audit_table = df_master_dictionary.join(
        df_real_counts,
        "zone_key",
        "left"
    ).select(
        "zone_key",
        "semantic_name",
        F.coalesce(F.col("n_obs_real"), F.lit(0)).alias("n_obs_real")
    ).orderBy(F.col("n_obs_real").desc())

    # --- 5. Final report ---
    print("\nReal-world census (67 live zones):")
    final_report_pd = df_final_audit_table.toPandas()
    display(final_report_pd)

    print("-" * 80)
    print("Audit complete. The real-world DNA is synchronized.")

except Exception as e:
    print(f"ERROR IN THE AUDIT: {e}")
    raise e


Starting forensic audit of reality...

Real-world census (67 live zones):


,zone_key,semantic_name,n_obs_real
0,Unassigned,unassigned_area,1366
1,P_25,roma_condesa_2,223
2,P_1,santa_fe_centro_comercial,169
3,P_9,santa_fe_ibero,150
4,P_32,carso_antara_miyana,122
5,C_1,terminal_1_aicm,111
6,P_13,santa_fe_quintana__sante_fe_patio,104
7,P_15,herradura_conscripto,99
8,P_8,ave_club_de_golf_lomas__interlomas_magnocentro__vialidad_de_la_barranca,96
9,P_12,tamarindos,82


--------------------------------------------------------------------------------
Audit complete. The real-world DNA is synchronized.


**Note — topological residuals (P_ vs. C_):** after running the density census on real data (step 3.6), 6 algorithmically-derived (C_) zones showed an unusually low observation count (N < 25), notably `C_18 (santa_fe_patio)` and `C_32 (tacubaya)` with N=1.

**Why this happens:** although the Master Coalesce's original goal was for the manually-drawn polygons to absorb most of the activity, these residuals are a "geo-statistical spillover" effect — manual polygons have rigid geometric boundaries, while HDBSCAN clusters by density; some offers land a few meters outside the hand-drawn polygon but are still part of the same real-world cluster the algorithm detects. Cases like `santa_fe_itesm` (N=6) or `santa_fe_abc` (N=8) are the periphery of high-demand zones the machine recognizes even where the human eye didn't draw a boundary.

**Decision: preserve these 67 zones, residuals included** — the GAN learned these micro-points exist, so removing them now would create an inconsistency between the trained model and the output dictionary; and they represent genuine market edge cases useful for testing model sensitivity in low-visibility zones later.

Topology considered finalized and validated — proceeding to the full denormalization of the million-row manifold.

## Phase 4 — Dictionary persistence

Persists the validated product, dropoff, and pickup dictionaries to local parquet files, ready to be joined onto the manifold.

#### 4.1 — Unified dictionary forge & persistence

Builds and saves the product hierarchy dictionary and the dropoff zone dictionary (P_ + C_ + Unassigned, filtered to the 67 zones that survive in the manifold).

In [10]:
# --- 1. Local paths ---
DICT_DIR = "/workspaces/pienza/data/dumped_files"
os.makedirs(DICT_DIR, exist_ok=True)

PROD_DICT_PATH = f"{DICT_DIR}/dim_product_hierarchy.parquet"
ZONE_DICT_PATH = f"{DICT_DIR}/dim_dropoff_zone_dictionary.parquet"

print("Forging the Pienza metadata vault (local)...")
print("-" * 65)

# --- 2. Product dictionary (1-2-3 hierarchy) ---
products_dna = [(1, "X"), (2, "Mid-Tier"), (3, "Premium")]
df_dim_products = spark.createDataFrame(products_dna, ["id_p", "product_name"])

# --- 3. Geography dictionary (manual names + survivors) ---
# A. Manual names (P_ axis)
df_p = spark.createDataFrame(p_axis_zone_names, ["id", "semantic_name"]) \
            .withColumn("zone_key", F.concat(F.lit("P_"), F.col("id").cast("string"))) \
            .select("zone_key", "semantic_name")

# B. Machine names (C_ axis, from real-data BigQuery)
print("Fetching cluster metadata from BigQuery...")
try:
    df_c = spark.read.format("bigquery") \
            .option("table", f"{GCP_PROJECT_ID}.pienza_mini.silver_palette") \
            .option("parentProject", GCP_PROJECT_ID) \
            .load() \
            .select("dropoff_hdbscan_id", "dropoff_hdbscan_name") \
            .distinct() \
            .filter(F.col("dropoff_hdbscan_id") != -1) \
            .withColumn("zone_key", F.concat(F.lit("C_"), F.col("dropoff_hdbscan_id").cast("string"))) \
            .select("zone_key", F.col("dropoff_hdbscan_name").alias("semantic_name"))
except Exception as e:
    print(f"Error connecting to BigQuery: {e}. Using an empty fallback for the C_ axis.")
    df_c = spark.createDataFrame([], df_p.schema)

# C. Unassigned & union
df_u = spark.createDataFrame([("Unassigned", "unassigned_area")], ["zone_key", "semantic_name"])
df_full_dict = df_p.union(df_c).union(df_u)

# D. Survival filter (only the 67 that exist in manifold V8)
print("Filtering by survival in the manifold...")
df_active_keys = df_manifold.select("dropoff_zone_id").distinct()
df_dim_zones = df_full_dict.join(df_active_keys, df_full_dict.zone_key == df_active_keys.dropoff_zone_id, "inner") \
                           .select("zone_key", "semantic_name")

# --- 4. Physical persistence in the workspace ---
try:
    print(f"Saving dictionaries to: {DICT_DIR} ...")
    df_dim_products.toPandas().to_parquet(PROD_DICT_PATH, index=False)
    df_dim_zones.toPandas().to_parquet(ZONE_DICT_PATH, index=False)

    print("\nMetadata vault updated.")
    print(f"   Products: {df_dim_products.count()} tiers (Pienza DNA)")
    print(f"   Geography: {df_dim_zones.count()} zones (manual names + clusters)")
except Exception as e:
    print(f"Error persisting locally: {e}")

print("-" * 65)


Forging the Pienza metadata vault (local)...
-----------------------------------------------------------------
Fetching cluster metadata from BigQuery...
Filtering by survival in the manifold...
Saving dictionaries to: /workspaces/pienza/data/dumped_files ...



Metadata vault updated.
   Products: 3 tiers (Pienza DNA)
   Geography: 67 zones (manual names + clusters)
-----------------------------------------------------------------


#### 4.2 — Pickup dictionary forge (mirror of dropoff)

Builds the origin-zone dictionary the same way, filtered by the manifold's `pickup_zone_id` instead of `dropoff_zone_id`.

In [11]:
# --- 1. Local path ---
# Same directory defined in 4.1
PICKUP_DICT_PATH = "/workspaces/pienza/data/dumped_files/dim_pickup_zone_dictionary.parquet"

print("Forging the pickup (origin) dictionary...")
print("-" * 65)

# --- 2. Reconstruct the universe if needed (kernel-restart safety) ---
if 'df_full_dict' not in locals():
    print("   -> Reconstructing the zone universe (P_ + C_) from session variables...")
    try:
        df_full_dict = df_p.union(df_c).union(df_u)
    except NameError:
        print("CRITICAL ERROR: run 4.1 first to define the P_ and C_ axes.")
        raise

# --- 3. Survival filter (pickup) ---
print("Identifying active origin zones in the manifold...")

if "pickup_zone_id" in df_manifold.columns:
    # Unique origin IDs across the million rows
    df_active_pickup = df_manifold.select("pickup_zone_id").distinct()

    # Join: map the full universe against the real origin IDs
    df_dim_pickup = df_full_dict.join(
        df_active_pickup,
        df_full_dict.zone_key == df_active_pickup.pickup_zone_id,
        "inner"
    ).select("zone_key", "semantic_name")

    # --- 4. Persistence ---
    try:
        print(f"Saving origin dictionary to: {PICKUP_DICT_PATH}")
        df_dim_pickup.toPandas().to_parquet(PICKUP_DICT_PATH, index=False)

        print("\nPickup dictionary secured.")
        print(f"   Total: {df_dim_pickup.count()} origin semantic identities.")
        print("   Ready for the triple star schema join.")
    except Exception as e:
        print(f"Error persisting locally: {e}")

else:
    print("CRITICAL ERROR: 'pickup_zone_id' not found in the manifold.")
    print("   Columns detected:", df_manifold.columns)

print("-" * 65)


Forging the pickup (origin) dictionary...
-----------------------------------------------------------------
Identifying active origin zones in the manifold...
Saving origin dictionary to: /workspaces/pienza/data/dumped_files/dim_pickup_zone_dictionary.parquet



Pickup dictionary secured.
   Total: 43 origin semantic identities.
   Ready for the triple star schema join.
-----------------------------------------------------------------


## Phase 5 — Triple star schema join

Joins the manifold against the three validated dictionaries (product, dropoff, pickup) to inject semantic names, then audits name coverage.

#### 5.1 — Triple broadcast join (denormalization)

Broadcast-joins the small dictionaries against the ~1M-row manifold (product, then dropoff, then pickup), and reports the resulting name coverage.

In [12]:
print("Injecting semantic intelligence into the synthetic manifold...")
print("-" * 80)

# --- 1. Local artifact paths ---
DICT_DIR = "/workspaces/pienza/data/dumped_files"
PATH_MANIFOLD = "/workspaces/pienza/data/dumped_files/260426_cGAN_manifold_v8.parquet"

PATH_DIM_PROD = f"{DICT_DIR}/dim_product_hierarchy.parquet"
PATH_DIM_DROP = f"{DICT_DIR}/dim_dropoff_zone_dictionary.parquet"
PATH_DIM_PICK = f"{DICT_DIR}/dim_pickup_zone_dictionary.parquet"

# --- 2. Read the data ---
print("   -> Loading dictionaries and manifold...")
try:
    df_dim_prods = spark.read.parquet(PATH_DIM_PROD)
    df_dim_drop = spark.read.parquet(PATH_DIM_DROP)
    df_dim_pick = spark.read.parquet(PATH_DIM_PICK)

    df_facts = spark.read.parquet(PATH_MANIFOLD)
    print(f"   Data loaded. Manifold detected with {df_facts.count():,} records.")

except Exception as e:
    print("ERROR: local files not found. Check cells 1.2, 4.1, and 4.2.")
    raise e

# --- 3. Triple broadcast join (memory optimization) ---
# Broadcast is used because the dictionaries are small and the manifold is massive.
print("Running the massive denormalization (3 dimensions)...")

# A. Dimension: product (X, Mid, Premium)
df_step1 = df_facts.join(
    F.broadcast(df_dim_prods),
    df_facts.product_category_fk == df_dim_prods.id_p,
    "left"
).drop("id_p")

# B. Dimension: dropoff (destination)
df_step2 = df_step1.join(
    F.broadcast(df_dim_drop),
    df_step1.dropoff_zone_id == df_dim_drop.zone_key,
    "left"
).drop("zone_key").withColumnRenamed("semantic_name", "dropoff_name")

# C. Dimension: pickup (origin)
df_final = df_step2.join(
    F.broadcast(df_dim_pick),
    df_step2.pickup_zone_id == df_dim_pick.zone_key,
    "left"
).drop("zone_key").withColumnRenamed("semantic_name", "pickup_name")

# --- 4. Coverage validation ---
print("\nSemantic integrity audit:")
print("-" * 80)

total = df_final.count()
named_pickup = df_final.filter(F.col("pickup_name").isNotNull()).count()
named_dropoff = df_final.filter(F.col("dropoff_name").isNotNull()).count()

print(f"Pickup coverage:  {named_pickup:,} / {total:,} ({(named_pickup / total) * 100:.2f}%)")
print(f"Dropoff coverage: {named_dropoff:,} / {total:,} ({(named_dropoff / total) * 100:.2f}%)")

# Random sample for a visual check
print("\nEnriched manifold sample:")
df_final.select(
    "product_name",
    "pickup_name",
    "dropoff_name",
    "upfront_fare"
).filter(F.col("pickup_name").isNotNull()) \
 .orderBy(F.rand()) \
 .show(15, truncate=False)

# --- 5. Cache for the downstream audits ---
df_final.cache()
print("Status: the master dataset is ready in the Spark engine.")


Injecting semantic intelligence into the synthetic manifold...
--------------------------------------------------------------------------------
   -> Loading dictionaries and manifold...
   Data loaded. Manifold detected with 1,010,001 records.
Running the massive denormalization (3 dimensions)...

Semantic integrity audit:
--------------------------------------------------------------------------------
Pickup coverage:  1,010,001 / 1,010,001 (100.00%)
Dropoff coverage: 1,010,001 / 1,010,001 (100.00%)

Enriched manifold sample:
+------------+------------------------------------------------+------------------------------------------------+------------+
|product_name|pickup_name                                     |dropoff_name                                    |upfront_fare|
+------------+------------------------------------------------+------------------------------------------------+------------+
|Premium     |anzures                                         |rios                     

## Phase 6 — Upload & validation

Writes the enriched manifold to BigQuery natively via the Spark connector, then validates it in the cloud with two independent audits (route control stats, and geospatial purity against the master polygon file).

#### 6.1 — Write to BigQuery (segmented upload, corrected indexing)

Writes `df_final` to `pienza_big.synthetic_manifold_v8_enriched` in 250k-row chunks via the `google.cloud.bigquery` client. Uses `rdd.zipWithIndex()` for a true sequential global index across all partitions, instead of `F.monotonically_increasing_id()` (whose per-partition offset gaps were the root cause of the original bug: only partition 0's rows ever matched a chunk's filter range, so most of the manifold silently never got uploaded).

In [13]:
from pyspark.sql import Row

FULL_TABLE_ID = f"{GCP_PROJECT_ID}.{BQ_DATASET}.synthetic_manifold_v8_enriched"

# zipWithIndex gives a true sequential 0..N-1 index across all partitions,
# unlike monotonically_increasing_id() (which leaves large gaps between partitions).
print("Indexing manifold for chunked upload...")
rdd_indexed = df_final.rdd.zipWithIndex().map(
    lambda row_idx: Row(**row_idx[0].asDict(), row_idx=row_idx[1])
)
df_indexed = spark.createDataFrame(rdd_indexed)

client = bigquery.Client(project=GCP_PROJECT_ID)

total_rows = df_indexed.count()
chunk_size = 250000
start = 0

print(f"Starting upload of {total_rows:,} records...")

while start < total_rows:
    end = start + chunk_size
    print(f"Uploading segment: {start:,} to {end:,}...")

    # Filter the chunk and bring it down to Pandas
    pdf_chunk = df_indexed.filter((F.col("row_idx") >= start) & (F.col("row_idx") < end)).toPandas()
    pdf_chunk = pdf_chunk.drop(columns=['row_idx'])

    # First pass overwrites, the rest append
    disposition = "WRITE_TRUNCATE" if start == 0 else "WRITE_APPEND"
    job_config = bigquery.LoadJobConfig(write_disposition=disposition)

    job = client.load_table_from_dataframe(pdf_chunk, FULL_TABLE_ID, job_config=job_config)
    job.result()

    start += chunk_size
    print("   Segment complete.")

print("Done. The V8 manifold is in the cloud.")
spark.stop()


Indexing manifold for chunked upload...


Starting upload of 1,010,001 records...
Uploading segment: 0 to 250,000...


   Segment complete.
Uploading segment: 250,000 to 500,000...


   Segment complete.
Uploading segment: 500,000 to 750,000...


   Segment complete.
Uploading segment: 750,000 to 1,000,000...


   Segment complete.
Uploading segment: 1,000,000 to 1,250,000...


   Segment complete.
Done. The V8 manifold is in the cloud.


#### 6.2 — Cloud audit: route control stats

Validates geographic control directly in BigQuery: what share of trips got both endpoints named, how many are intra-zone, and the top 15 routes by volume.

In [14]:
client = bigquery.Client(project=GCP_PROJECT_ID)
TABLE_ID = f"{GCP_PROJECT_ID}.{BQ_DATASET}.synthetic_manifold_v8_enriched"

print("Extracting the heart of Pienza (cloud analysis)...")
print("-" * 60)

# 1. Overall control metrics (single pass over the data)
query_stats = f"""
SELECT
    COUNT(*) as total_filas,
    COUNTIF(pickup_name != 'unassigned_area' AND dropoff_name != 'unassigned_area') as total_endogenos,
    COUNTIF(pickup_name = dropoff_name AND pickup_name != 'unassigned_area') as total_intra_zonales
FROM `{TABLE_ID}`
"""

stats = client.query(query_stats).to_dataframe()

total = stats['total_filas'][0]
endogenos = stats['total_endogenos'][0]
intra = stats['total_intra_zonales'][0]
tasa_control = (endogenos / total) * 100

print("Geographic control summary:")
print(f"Total trips in manifold:     {total:,}")
print(f"100%-identified trips:      {endogenos:,}")
print(f"Master control rate:        {tasa_control:.2f}%")
print(f"Intra-zone trips:           {intra:,} ({(intra / endogenos) * 100:.1f}% of endogenous trips)")
print("-" * 60)

# 2. Top 15 routes
query_rutas = f"""
SELECT
    pickup_name,
    dropoff_name,
    COUNT(*) as conteo,
    ROUND(COUNT(*) * 100 / {endogenos}, 2) as share_porcentaje
FROM `{TABLE_ID}`
WHERE pickup_name != 'unassigned_area' AND dropoff_name != 'unassigned_area'
GROUP BY 1, 2
ORDER BY conteo DESC
LIMIT 15
"""

df_rutas = client.query(query_rutas).to_dataframe()

print("\nTop 15 master routes (end-to-end control):")
display(df_rutas)

print("\nAudit finished. Cloud data is consistent with Pienza's DNA.")


Extracting the heart of Pienza (cloud analysis)...
------------------------------------------------------------
Geographic control summary:
Total trips in manifold:     1,010,001
100%-identified trips:      587,051
Master control rate:        58.12%
Intra-zone trips:           27,290 (4.6% of endogenous trips)
------------------------------------------------------------

Top 15 master routes (end-to-end control):


,pickup_name,dropoff_name,conteo,share_porcentaje
0,tamarindos,santa_fe_ibero,6218,1.06
1,santa_fe_ibero,santa_fe_centro_comercial,5179,0.88
2,anzures,polanco_parque_lincoln,4903,0.84
3,anzures,anahuac_1__bahias__frontera_polanco,4733,0.81
4,anzures,terminal_1_aicm,4510,0.77
5,anzures,rios,4388,0.75
6,anahuac_1__bahias__frontera_polanco,terminal_2_aicm,4266,0.73
7,santa_fe_quintana__sante_fe_patio,santa_fe_quintana__sante_fe_patio,4061,0.69
8,santa_fe_ibero,santa_fe_ibero,3947,0.67
9,rios,roma_condesa_2,3761,0.64



Audit finished. Cloud data is consistent with Pienza's DNA.


#### 6.3 — Geospatial validator (BigQuery native)

Cross-checks the manifold against the master polygon file (`poly.geojson`): what fraction of trips have both endpoints inside an officially recognized polygon, and the top 10 certified routes.

In [15]:
PATH_GEOJSON = "/workspaces/pienza/assets/poly.geojson"
TABLE_ID = f"{GCP_PROJECT_ID}.{BQ_DATASET}.synthetic_manifold_v8_enriched"
client = bigquery.Client(project=GCP_PROJECT_ID)

print("Loading master polygons from assets...")

try:
    # 1. Extract names from the local GeoJSON
    gdf_polys = gpd.read_file(PATH_GEOJSON)
    zonas_oficiales = list(set(gdf_polys['name'].unique()))

    # Prepare the list for SQL: ['zone1', 'zone2'] -> "'zone1', 'zone2'"
    zonas_sql = ", ".join([f"'{z}'" for z in zonas_oficiales])

    print(f"Local map: {len(zonas_oficiales)} zones detected.")

    # 2. Validation query (all compute happens in BigQuery)
    query_pureza = f"""
    WITH stats AS (
        SELECT
            COUNT(*) as total_manifold,
            COUNTIF(pickup_name IN ({zonas_sql}) AND dropoff_name IN ({zonas_sql})) as total_puros
        FROM `{TABLE_ID}`
    )
    SELECT *, (total_puros / total_manifold) * 100 as eficiencia FROM stats
    """

    res = client.query(query_pureza).to_dataframe()

    total_manifold = res['total_manifold'][0]
    total_puros = res['total_puros'][0]
    eficiencia = res['eficiencia'][0]

    print("\nGeographic validation results (cloud mode):")
    print("-" * 55)
    print(f"Records in BigQuery: {total_manifold:,}")
    print(f"'Map-pure' records (validated): {total_puros:,}")
    print(f"Simulation efficiency: {eficiencia:.2f}%")
    print("-" * 55)

    # 3. Top 10 certified routes
    query_top = f"""
    SELECT pickup_name, dropoff_name, COUNT(*) as conteo
    FROM `{TABLE_ID}`
    WHERE pickup_name IN ({zonas_sql}) AND dropoff_name IN ({zonas_sql})
    GROUP BY 1, 2
    ORDER BY conteo DESC
    LIMIT 10
    """

    df_top_puros = client.query(query_top).to_dataframe()

    print("\nTop 10 routes certified by poly.geojson:")
    display(df_top_puros)

except Exception as e:
    print(f"ERROR IN CLOUD VALIDATION: {e}")


Loading master polygons from assets...
Local map: 71 zones detected.

Geographic validation results (cloud mode):
-------------------------------------------------------
Records in BigQuery: 1,010,001
'Map-pure' records (validated): 185,342
Simulation efficiency: 18.35%
-------------------------------------------------------

Top 10 routes certified by poly.geojson:


,pickup_name,dropoff_name,conteo
0,tamarindos,santa_fe_ibero,6218
1,santa_fe_ibero,santa_fe_centro_comercial,5179
2,anzures,polanco_parque_lincoln,4903
3,anzures,rios,4388
4,santa_fe_ibero,santa_fe_ibero,3947
5,rios,roma_condesa_2,3761
6,tamarindos,santa_fe_centro_comercial,3712
7,herradura_conscripto,herradura_conscripto,3710
8,tamarindos,ahuehuetes_sur,3572
9,rios,roma_condesa_1,3552


---

## Appendix — variable journey

```mermaid
flowchart TD
    local_pq["local parquet\n260426_cGAN_manifold_v8.parquet"] --> df_manifold["df_manifold\n(Spark DataFrame, Phase 1)"]

    bq_zones["BigQuery: real zone geography"] --> lookup["Master zone lookup table\n(P_ / C_ axis reconciliation)"]
    df_manifold --> purge["Geographic purge + ghost recovery\n(active keys from df_manifold, 67 survivors of 89 zones)"]
    lookup --> purge
    purge --> df_full_dict["df_full_dict\n(unified dropoff dictionary)"]
    df_full_dict --> df_pickup_dict["Pickup dictionary\n(mirror of dropoff)"]

    df_full_dict --> dict_parquets["Dictionaries persisted to local parquet\n(product / dropoff / pickup, Phase 4)"]
    df_pickup_dict --> dict_parquets

    local_pq --> df_facts["df_facts\n(manifold re-read fresh, Phase 5.1)"]
    dict_parquets --> df_dims["df_dim_prods / df_dim_drop / df_dim_pick\n(dictionaries re-read fresh, Phase 5.1)"]

    df_facts --> join["Triple broadcast join\n(manifold + product + zone dicts)"]
    df_dims --> join
    join --> df_final["df_final\n(denormalized, semantic names injected)"]

    df_final --> upload["Segmented BigQuery upload\n(zipWithIndex, sequential chunking)"]
    upload --> table["synthetic_manifold_v8_enriched"]
    table --> validate["Route-control + geospatial\nnative BigQuery audits"]

    classDef default stroke:#21918c,stroke-width:2px;
    linkStyle default stroke:#21918c,stroke-width:2px;
```